In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대해 최빈단어시각화와 유사도분석
- (1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각각 500건씩 검색하여 백업(data/quiz/naver.csv)
    * 파일 내용 : query, no, title, link, description, total_text(title + ' ' + description)
- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
    * 파일 내용 : query, no, token, pos
- (3) 명사만 추출(naver_pos_nouns.csv)
    * query, token, pos
- (4) 빈도분석 백업(naver_pos_nouns_count.csv)
    * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도 시각화(워드클라우드, Text.plot)
    * 이미지 저장
- (6) 단어간 거리 분석(Word2Vec)

## 1. 네이버 open API 활용하여 검색 추출
- query, no, title, link, description, total_text(title + ' ' + description)

In [2]:
# .env가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
# 네이버 개발자 센터에 있는 소스를 가져오기
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog.json?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 16:52:37 +0900",
	"total":2735034,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"description":"첨가본 동궁원. 예쁜 꽃들 만나고 좋아하는 커피열매도 보고 비맞으면서 노천온천 잘먹고 잘쉬고 행복한 <b>경주 여행<\/b>.",
			"bloggername":"달빛의 블로그",
			"bloggerlink":"https:\/\/lje77777.tistory.com\/",
			"postdate":"20210307"
		},
		{
			"title":"<b>경주여행<\/b> 황리단길 보문단지 왕릉탐방 데이트코스",
			"link":"https:\/\/a-kdh.tistory.com\/340",
			"description":"https:\/\/youtu.be\/_nZzijKooIk 밤의 #황리단길 은 참 예쁘고 마음 한켠이 낭만으로 차오르는 경상도에서 흔치 않은 젊은거리 바로 옆은 #<b>경주<\/b>유적지 에 오래된 한옥 #천년고도 <b>경주<\/b>와는 사뭇 어울리면서도... ",
			"bloggername":"최고의 순간은 지금이다.",
			"bloggerlink":"https:\/\/a-kdh.tistory.com\/",
			"postdate":"20211109"
		},
		{
			"title":"<b>경주여행<\/b>밀면",
			"link":"https:\/\/imsun202.tistory.com\/15715529",
			"description":"<b>경주<\/b>에서 먹어본 &quot;<b>경주여행<\/b>밀면&quot; 맛보기 가격도 착해 5000원 비빔밀면 물밀면도 5000원 더해먹는 떡갈비는 2000원 12,000원 밥상 그리고 뜨거운 육수까지 맛평